# Does the kappa residual predict the excursion cutoff ratio, rule by rule?

## Question and why it matters
Matched-area pairs (`experiments/archive/2026-10-kappa-matched-area`) suggested the higher-kappa rule has the smaller ln(L_99.9/R_eq) in ~80% of pairs. This experiment tests the rule-level version with all 44 rules, no matching, kappa/ln A from one seed half and ln_ratio from the other, with a random rule intercept (own REML). See `README.md` (registration) and `NOTEBOOK.md`.

## Assumptions
Seed halves are independent replicates (untested); a linear kappa-on-ln A residual defines 'kappa given area' (untested); rule-cluster bootstrap with duplicated rules as distinct groups is adequate (checked by synthetic coverage 0.90); the Gaussian homoscedastic model is adequate for ln_ratio (**false**: heavy-tailed, see below).


## Provenance


In [1]:
import hashlib, subprocess, datetime, json
from pathlib import Path
SEED = 20261016  # used by all scripts; recorded in README
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').exists())
EXP = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'NOTEBOOK.md').exists())
INPUTS = [EXP/'results'/f for f in ('calibration.txt','fit.txt','fit.json','posthoc.txt')]
assert all(p.exists() for p in INPUTS)
sha = subprocess.run(['git','rev-parse','--short','HEAD'],cwd=ROOT,capture_output=True,text=True).stdout.strip()
print('git', sha, '| run', datetime.datetime.now().isoformat(timespec='minutes'), '| seed', SEED)
for p in INPUTS: print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(ROOT))
print((EXP/'results'/'fit.txt').read_text().splitlines()[1])  # sha of the T1b seeds file used upstream


git 269288e | run 2026-10-16T00:33 | seed 20261016
d47044732a62657e experiments/2026-10-15-kappa-cutoff-mixed-effects/results/calibration.txt
8a599f3884b30b9d experiments/2026-10-15-kappa-cutoff-mixed-effects/results/fit.txt
d4defcc9dbdf910f experiments/2026-10-15-kappa-cutoff-mixed-effects/results/fit.json
aacc1c7c33a0dffe experiments/2026-10-15-kappa-cutoff-mixed-effects/results/posthoc.txt
# input sha256[:16] e813c6f21f63421c | rows 1399


## Data and method
`scripts/mixed.py` (REML + split pipeline), `scripts/calibrate.py`, `scripts/fit.py` (registered), `scripts/posthoc.py` (exploratory). Input: committed `t1b_cutoff_seeds.csv` of the T1/T1b experiment (k=22, 25; 44 rules; 6-8 seeds). Heavy computation is in those scripts; this notebook only reads their committed outputs.

```bash
uv run experiments/archive/2026-10-kappa-cutoff-mixed-effects/scripts/calibrate.py
uv run experiments/archive/2026-10-kappa-cutoff-mixed-effects/scripts/fit.py
uv run experiments/archive/2026-10-kappa-cutoff-mixed-effects/scripts/posthoc.py
```


## Results: calibration gate (criterion C)


In [2]:
print((EXP/'results'/'calibration.txt').read_text())


# calibrate.py; commit 997c9b2da6efa526fc6c5a57104d103e1345e916
## noise-free covariates, br=-0.5: 300 datasets, bootstrap B=200
  br estimate mean -0.489 (truth -0.5), sd 0.066; ratio est/truth 0.977
  s_u mean 0.297 (truth 0.30), s_e mean 0.130 (truth 0.13)
  bootstrap 95% CI covers the true br: 0.903;  CI excludes 0: 1.000
## noise-free covariates, br=0 (null): 300 datasets, bootstrap B=200
  br estimate mean 0.000 (truth 0.0), sd 0.049; ratio est/truth n/a
  s_u mean 0.301 (truth 0.30), s_e mean 0.129 (truth 0.13)
  bootstrap 95% CI covers the true br: 0.923;  CI excludes 0: 0.077
## noisy covariates (seed sd lnk 0.089, lnA 0.10), br=-0.5: 300 datasets, bootstrap B=200
  br estimate mean -0.447 (truth -0.5), sd 0.077; ratio est/truth 0.895
  s_u mean 0.363 (truth 0.30), s_e mean 0.130 (truth 0.13)
  bootstrap 95% CI covers the true br: 0.787;  CI excludes 0: 1.000



Shows the estimator recovers a known br (ratio 0.98), s_u, s_e, and rejects 7.7% under the null; the CI coverage 0.903 is at the edge of the registered band. It does not show the model fits the real data: the synthetic noise is Gaussian and homoscedastic.


## Results: registered fit


In [3]:
j = json.loads((EXP/'results'/'fit.json').read_text())
crit = dict(Std=-0.30, dR2=0.05, frac=0.90, p=0.01)
print(f"{'':4s}{'br':>8s}{'Std':>8s}{'Std CI':>18s}{'dR2':>8s}{'frac>0':>8s}{'p_perm':>8s}{'StdB':>8s}")
for k in ('22','25'):
    r = j[k]; print(f"k={k} {r['br']:8.3f}{r['Std']:8.3f}  [{r['ci_Std'][0]:6.3f},{r['ci_Std'][1]:6.3f}]{r['dR2']:8.3f}{r['frac_dR2_pos']:8.3f}{r['p_perm']:8.3f}{r['StdB']:8.3f}")
print('registered: Std <= -0.30, dR2 >= 0.05, frac>0 >= 0.90, p <= 0.01')
print('pooled over k:', {a: j['pooled'][a] for a in ('br','Std','ci_br')})
print('P1', all(j[k]['Std'] <= -0.30 and j[k]['ci_br'][1] < 0 for k in ('22','25')),
      'P2', all(j[k]['dR2'] >= 0.05 and j[k]['frac_dR2_pos'] >= 0.90 for k in ('22','25')),
      'P3', all(j[k]['p_perm'] <= 0.01 for k in ('22','25')),
      'P4', j['pooled']['br'] < 0 and j['pooled']['ci_br'][1] < 0 and j['pooled']['Std'] <= -0.30)


          br     Std            Std CI     dR2  frac>0  p_perm    StdB
k=22   -0.415  -0.275  [-0.436,-0.091]   0.118   0.875   0.020  -0.374
k=25   -0.570  -0.194  [-0.482,-0.072]   0.073   0.866   0.096  -0.313
registered: Std <= -0.30, dR2 >= 0.05, frac>0 >= 0.90, p <= 0.01
pooled over k: {'br': -0.013950091931779854, 'Std': -0.006274791657703334, 'ci_br': [-0.8896540874063015, 0.2869464829892926]}
P1 False P2 False P3 False P4 False


The sign is negative and the bootstrap interval for br excludes 0 at both k, but the standardised effect (-0.28, -0.19) is below the registered -0.30 threshold, the held-out gain is not reliably positive (0.87 of replicates), the permutation p is 0.02 and 0.10, and the pooled slope is about 0. **The registered hypothesis failed.** It does not show there is no association: the sign agrees in every analysis, in both split directions, and between-split sd of br (0.075, 0.161) is as large as the gap to the threshold.


The full text output (with s_u, s_e, split-sd, shuffle control):


In [4]:
print((EXP/'results'/'fit.txt').read_text())


# fit.py; commit 997c9b2da6efa526fc6c5a57104d103e1345e916
# input sha256[:16] e813c6f21f63421c | rows 1399
rules 44; seeds per rule/k min 6 max 8
parity split: X = even seeds, Y = odd seeds; direction A = covariates from X, ln_ratio from Y
## k=22
direction A (parity): br=-0.4152 (REML SE 0.1632) bA=-1.0116 b0=4.4023; ysd(rule means)=1.5108 -> Std=-0.275
  s_u=0.9294 s_e=1.0576; s_u^2 zA-only 1.0131 -> with zr 0.8638 (share removed 0.147)
  out-of-sample R2 (44 rule means, frozen transform): M0 0.564, M1 0.681, dR2 +0.118
direction B (parity, sensitivity): br=-0.5455 (SE 0.1232) Std=-0.374; dR2 +0.055
rule-cluster bootstrap B=1000, fresh split per replicate: br 95% [-0.6622, -0.1347]; Std 95% [-0.436, -0.091]; dR2 95% [-0.101, +0.155], frac dR2>0 0.875
random balanced splits R=500 (44 rules fixed): br mean -0.4199 sd 0.0751 (between-split sd), frac<0 1.000; Std mean -0.279 sd 0.046; dR2 mean +0.051 sd 0.046
permutation of zr across rules (1000): null br sd 0.1774; two-sided p = 0.0200


## Caveats and exploratory follow-up (not registered)


In [5]:
print((EXP/'results'/'posthoc.txt').read_text())


# posthoc.py (exploratory); commit 269288eb62bf6b6ec125a3b420dd899057858be5
# input sha256[:16] e813c6f21f63421c
k=22 quiet: rules 27; A: br=-0.2073 Std=-0.428 boot Std95 [-0.673,-0.188]; s_u=0.329 s_e=0.103; dR2 +0.254 frac>0 0.996; perm p 0.0070; B: Std=-0.510
k=22 rank: rules 44; A: br=-0.2252 Std=-0.265 boot Std95 [-0.433,-0.081]; s_u=0.484 s_e=0.602; dR2 +0.090 frac>0 0.883; perm p 0.0200; B: Std=-0.323
k=25 quiet: rules 29; A: br=-0.3759 Std=-0.392 boot Std95 [-0.587,-0.221]; s_u=0.325 s_e=0.125; dR2 +0.173 frac>0 1.000; perm p 0.0010; B: Std=-0.420
k=25 rank: rules 44; A: br=-0.1849 Std=-0.197 boot Std95 [-0.424,-0.075]; s_u=0.602 s_e=0.406; dR2 +0.053 frac>0 0.824; perm p 0.0539; B: Std=-0.247



Among the 27-29 rules with stable cutoffs (seed sd of ln_ratio <= 0.5) every registered threshold would be met; ranks over all rules would not. This subset was chosen after the failure, on the same data, so it is hypothesis-generating only. REML s_e is ~1 against a median seed sd of 0.13: a minority of bursty rules makes the Gaussian homoscedastic model misspecified. The pooled slope (~0) is undiagnosed. Windows k=23, 24 were not used.

## What this changes, next steps
No T2 on this basis. Next (to be registered first): replicate the quiet-rule subset on the k=23, 24 windows, fit a heteroscedastic or robust model for all rules, and add a two-k synthetic calibration to explain the pooled result.
